# 1D CNN-A — Visualise: Thumbnail Grid

Runs the full data pipeline and renders `N_SAMPLE` windows as a **thumbnail grid** —
each 64 × 14-pixel window downsampled to `THUMB_PX` × `THUMB_PX` pixels via
Lanczos resampling, arranged `GRID_COLS_C` thumbnails per row.

Useful for spotting repeating texture clusters across the dataset at a glance.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

from datetime import date

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
csv_path = os.path.join(DATA_DIR, SYMBOL, f"{TIMEFRAME}.csv")
df = pd.read_csv(csv_path, parse_dates=["timestamp"], nrows=MAX_BARS)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df):,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
max_bars_display = 'all' if MAX_BARS is None else f'{MAX_BARS:,}'
print(f"(MAX_BARS={max_bars_display})")
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
df = df.drop_duplicates(subset=["timestamp"])
df = df.dropna()

df.isnull().sum()
df.duplicated(subset=["timestamp"]).sum()
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# EMAs
df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

# MACD components
df["macd_12"] = df["close"].ewm(span=12, adjust=False).mean()
df["macd_26"] = df["close"].ewm(span=26, adjust=False).mean()

# MACD line
df["macd"] = df["macd_12"] - df["macd_26"]

# MACD signal line (9 EMA of MACD)
df["macd_9"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD histogram (optional but commonly used)
df["macd_hist"] = df["macd"] - df["macd_9"]

# Candle details
df["body"] = df["close"] - df["open"]
df["upper_wick"] = df["high"] - df[["open", "close"]].max(axis=1)
df["lower_wick"] = df[["open", "close"]].min(axis=1) - df["low"]



# other
df["return"] = df["close"].pct_change()
df["vol_return"] = df["volume"].pct_change()
df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["volume_ratio"] = (
    df["volume"] /
    df["volume"].rolling(20).mean()
)

# display sample of new features
# df[["timestamp", "close", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

df[df["body"] != 0][
    [
        "timestamp",
        "close",
        "ema_9",
        "ema_21",
        "ema_50",
        "macd",
        "macd_9",
        "macd_hist",
        "body",
        "upper_wick",
        "lower_wick",
        "return",
        "vol_return",
        "log_return",
        "volume_ratio",
    ]
].tail()

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
df = df.dropna().reset_index(drop=True)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

# feature_cols is defined in config.py — edit it there to change which features are used
# scaler = StandardScaler()
# df[feature_cols] = scaler.fit_transform(df[feature_cols])

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
# Often better for financial data due to outliers

df[["timestamp", "open", "high", "low", "close", "volume", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
n_features = len(feature_cols)
print(f"Features ({n_features}):", feature_cols)
print("Data shape:", df[feature_cols].shape)

data = df[feature_cols].to_numpy(dtype=np.float32)

X_raw = np.lib.stride_tricks.sliding_window_view(
    data,
    window_shape=WINDOW_SIZE,
    axis=0
).transpose(0, 2, 1)   # → (N, WINDOW_SIZE, n_features)

print("X_raw shape:", X_raw.shape)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
diffs_sec = df["timestamp"].diff().dt.total_seconds().fillna(0).to_numpy()
gap_positions = np.where(diffs_sec > 300)[0]   # > 5 min between consecutive bars

valid_mask = np.ones(len(X_raw), dtype=bool)
for gp in gap_positions:
    lo = max(0, gp - WINDOW_SIZE + 1)
    hi = min(len(X_raw), gp + 1)
    valid_mask[lo:hi] = False

X_clean = X_raw[valid_mask]
print(f"Gap positions: {len(gap_positions)}")
print(f"Removed {(~valid_mask).sum():,} gap-spanning windows")
print(f"Clean windows: {X_clean.shape[0]:,}  shape: {X_clean.shape}")

## 9. Visualise Windows as Greyscale Tiles

Before training, it's worth seeing what the windowed data actually *looks like*.
Each of the 156,781 clean windows is a 64-bar × 14-feature matrix of scaled
floats. Rendering them as greyscale images (bright = high value, dark = low)
lets us visually inspect the dataset for structure, gaps, outliers, and feature
behaviour at a scale no table or line-chart can match.

Three views are provided — all use the same global [0 → 255] normalisation so
brightness is directly comparable across windows and across views:

| View | What you see | Best for |
|------|-------------|----------|
| A — Contact sheet | Each window at full 64×14 px resolution, tiled in a grid | Inspecting individual window texture and feature patterns |
| B — Heatmap strip | Each window flattened to one row, all windows stacked vertically | Spotting dataset-wide trends, session rhythms, and drift over time |
| C — Thumbnail grid | Each window compressed to 10×10 px, tiled densely | Getting a high-level overview of brightness/contrast across the whole sample |

Adjust `N_SAMPLE` in the setup cell below to control how many windows are rendered.

In [ ]:
from PIL import Image

# N_SAMPLE is defined in config.py — adjust it there to render more/fewer windows
sample = X_clean[:N_SAMPLE]                            # (N_SAMPLE, 64, 14)

# Percentile clipping: use p2/p98 instead of absolute min/max so that
# outlier spikes don't compress the bulk of values into a dark narrow band.
lo, hi         = np.percentile(sample, 2), np.percentile(sample, 98)
sample_clipped = np.clip(sample, lo, hi)
sample_u8      = ((sample_clipped - lo) / (hi - lo) * 255).astype(np.uint8)
N_SAMPLE       = len(sample_u8)  # reconcile: X_clean may have fewer rows than config target
# sample_u8: (N_SAMPLE, 64, 14) uint8 — values 0-255, outliers clamped

N_SAMPLE = len(sample_u8)  # reconcile: X_clean may have fewer rows than config target
print(f"Rendering {N_SAMPLE:,} windows  |  p2={lo:.3f}  p98={hi:.3f}  →  [0, 255]")

### View C — Thumbnail Grid (10 × 10 px per window)

Each 64×14 window is **downsampled to a 10×10 pixel thumbnail** using Lanczos
resampling — a high-quality filter that anti-aliases rather than simply dropping
pixels. The 100 resulting pixels capture the coarse spatial distribution of
values across the window (bright regions, dark regions, gradients) but not
fine detail.

Thumbnails are tiled left-to-right in rows of 100, forming a dense overview
of the full sample in a compact space.

**What to look for:**
- **Overall brightness distribution**: do most thumbnails cluster around
  mid-grey (balanced windows) or are many very bright/dark (trending or
  extreme market conditions)?
- **Clusters of similar-looking thumbnails**: groups of adjacent windows with
  the same gross texture — candidate pattern groups the autoencoder should
  discover. Randomness here is also informative.
- **Outlier blocks**: thumbnails that look starkly different from their
  neighbours (all-white, all-black, or sharply split) — near a data quality
  issue or a genuine market shock.
- **Texture gradient across the grid**: a slow shift in average brightness
  from top-left to bottom-right reflects the dataset's evolution over time,
  since windows are in chronological order.

This is the most information-compressed view but the most scannable at a glance.

In [ ]:
# THUMB_PX, GRID_COLS_C are defined in config.py
thumbs = np.stack([
    np.array(Image.fromarray(w, mode='L').resize((THUMB_PX, THUMB_PX), Image.LANCZOS))
    for w in sample_u8
])  # (N_SAMPLE, THUMB_PX, THUMB_PX)

n_rows_c = (N_SAMPLE + GRID_COLS_C - 1) // GRID_COLS_C
canvas_c = np.zeros((n_rows_c * THUMB_PX, GRID_COLS_C * THUMB_PX), dtype=np.uint8)
for i, block in enumerate(thumbs):
    r, c = divmod(i, GRID_COLS_C)
    canvas_c[r*THUMB_PX:(r+1)*THUMB_PX, c*THUMB_PX:(c+1)*THUMB_PX] = block

fig, ax = plt.subplots(figsize=(20, max(4, n_rows_c * THUMB_PX / 40)))
ax.imshow(canvas_c, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
ax.axis('off')
ax.set_title(
    f'View C — Thumbnail Grid  |  {N_SAMPLE:,} windows  |  '
    f'{THUMB_PX}×{THUMB_PX} px each  |  {GRID_COLS_C} per row  |  {n_rows_c} rows',
    fontsize=10
)
plt.tight_layout()
plt.show()